In [1]:
# Enable GPU (optional)
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

# Load data
from sklearn.model_selection import train_test_split
import openml
import numpy as np
import sklearn

dataset = openml.datasets.get_dataset(46915, download_data=True, download_qualities=True, download_features_meta_data=True)
X, y, categorical_indicator, attribute_names = dataset.get_data(target=dataset.default_target_attribute)

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(X_temp, y_temp, test_size=0.2, random_state=42)

# GRANDE (PyTorch)
from grande import GRANDE

params = {
    'depth': 5,
    'n_estimators': 1024,

    'learning_rate_weights': 0.001,
    'learning_rate_index': 0.01,
    'learning_rate_values': 0.05,
    'learning_rate_leaf': 0.05,
    'learning_rate_embedding': 0.02,  # used if embeddings are enabled

    # Embeddings (set True to enable)
    'use_category_embeddings': False,  # True to enable
    'embedding_dim_cat': 8,
    'use_numeric_embeddings': False,   # True to enable
    'embedding_dim_num': 8,
    'embedding_threshold': 1,          # low-cardinality split for categorical embeddings
    'loo_cardinality': 10,             # high-cardinality split for encoders

    'dropout': 0.2,
    'selected_variables': 0.8,
    'data_subset_fraction': 1.0,
    'bootstrap': False,
    'missing_values': False,

    'optimizer': 'adam',               # options: nadam, radam, adamw, adam
    'cosine_decay_restarts': False,
    'reduce_on_plateau_scheduler': True,
    'label_smoothing': 0.0,
    'use_class_weights': False,
    'focal_loss': False,
    'swa': False,
    'es_metric': True,  # AUC for binary, MSE for regression, val_loss for multiclass

    'epochs': 250,
    'batch_size': 256,
    'early_stopping_epochs': 50,

    'use_freq_enc': False,
    'use_robust_scale_smoothing': False,

    # Important: use problem_type, not objective
    'problem_type': 'binary',  # {'binary', 'multiclass', 'regression'}

    'random_seed': 42,
    'verbose': 2,
}

model_grande = GRANDE(params=params)
model_grande.fit(X=X_train, y=y_train, X_val=X_valid, y_val=y_valid)

# Predict
preds_grande = model_grande.predict_proba(X_test)

# Evaluate (binary)
accuracy = sklearn.metrics.accuracy_score(y_test, np.round(preds_grande[:, 1]))
f1 = sklearn.metrics.f1_score(y_test, np.round(preds_grande[:, 1]), average='macro')
roc_auc = sklearn.metrics.roc_auc_score(y_test, preds_grande[:, 1], average='macro')

print('Accuracy GRANDE:', accuracy)
print('F1 Score GRANDE:', f1)
print('ROC AUC GRANDE:', roc_auc)

Epoch 001 | TrainLoss: 0.5474 | ValLoss: 0.4453 | ValAcc: 0.8462 | ValAUC: 0.7365 | ValF1: 0.4584 | Time: 19.79s
Epoch 002 | TrainLoss: 0.3948 | ValLoss: 0.3909 | ValAcc: 0.8462 | ValAUC: 0.7867 | ValF1: 0.4584 | Time: 2.34s
Epoch 003 | TrainLoss: 0.3378 | ValLoss: 0.3544 | ValAcc: 0.8512 | ValAUC: 0.8047 | ValF1: 0.4911 | Time: 2.34s
Epoch 004 | TrainLoss: 0.2971 | ValLoss: 0.3175 | ValAcc: 0.8700 | ValAUC: 0.8547 | ValF1: 0.6081 | Time: 2.34s
Epoch 005 | TrainLoss: 0.2588 | ValLoss: 0.2883 | ValAcc: 0.8725 | ValAUC: 0.8808 | ValF1: 0.6248 | Time: 2.34s
Epoch 006 | TrainLoss: 0.2374 | ValLoss: 0.2643 | ValAcc: 0.8775 | ValAUC: 0.8942 | ValF1: 0.6479 | Time: 2.34s
Epoch 007 | TrainLoss: 0.2187 | ValLoss: 0.2469 | ValAcc: 0.8925 | ValAUC: 0.9007 | ValF1: 0.7140 | Time: 2.35s
Epoch 008 | TrainLoss: 0.2063 | ValLoss: 0.2278 | ValAcc: 0.9150 | ValAUC: 0.9068 | ValF1: 0.8006 | Time: 2.35s
Epoch 009 | TrainLoss: 0.1965 | ValLoss: 0.2169 | ValAcc: 0.9187 | ValAUC: 0.9129 | ValF1: 0.8102 | Tim

TypeError: Labels in y_true and y_pred should be of the same type. Got y_true=['No' 'Yes'] and y_pred=[0. 1.]. Make sure that the predictions provided by the classifier coincides with the true labels.